[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rglopesdf/mineracao-textos-2026/blob/main/notebooks/01_do_texto_ao_vetor.ipynb)

# Notebook 01 · Do texto ao vetor
**Mineração de Textos · Encontros 02 (17/11) e 03 (18/11) · Projeto Voz do Cliente, Etapa 1**

Neste notebook vamos sair de 132 mil avaliações reais de clientes da Americanas.com e chegar a vetores que um modelo consegue usar. Cada bloco segue o mesmo roteiro da aula: **problema → conceito → solução**.

| Bloco | Problema | Conceito | Solução |
|---|---|---|---|
| 1 | O texto vem sujo | Pré-processamento | Um pipeline com decisões justificadas |
| 2 | Contar palavras engana | BoW e TF-IDF | Termos que distinguem notas 1 e 5 |
| 3 | Mesmo problema, palavras diferentes | Embeddings e similaridade | Busca semântica e mapa 2D |

No fim, a seção **Etapa 1** traz o que o grupo precisa entregar.

> Dataset: B2W-Reviews01 (Real, Oshiro e Mafra, STIL 2019), licença CC BY-NC-SA 4.0. Uso acadêmico com crédito à B2W Digital.

## 0. Preparação do ambiente
No Colab, rode esta célula uma vez. Ela instala o que falta e baixa o modelo de português do spaCy.

In [ ]:
import importlib.util, subprocess, sys

# Instala apenas os pacotes que ainda não estão disponíveis no ambiente
faltando = [p for p, m in [("sentence-transformers", "sentence_transformers"), ("umap-learn", "umap")] if importlib.util.find_spec(m) is None]
if faltando:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)

# Carrega o modelo de português do spaCy (sem parser/NER para acelerar); baixa se faltar
import spacy
try:
    nlp = spacy.load("pt_core_news_sm", disable=["parser", "ner"])
except OSError:
    spacy.cli.download("pt_core_news_sm")
    nlp = spacy.load("pt_core_news_sm", disable=["parser", "ner"])

# Baixa a lista de stopwords do NLTK
import nltk
nltk.download("stopwords", quiet=True)
print("Ambiente pronto")

In [ ]:
# Bibliotecas base usadas ao longo do notebook
import os, re, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configurações de exibição das tabelas e estilo padrão dos gráficos
pd.set_option("display.max_colwidth", 140)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

## 1. Carregar e conhecer os dados
O CSV tem cerca de 50 MB e é lido direto do GitHub.

In [ ]:
import os
import pandas as pd

# Usa arquivo local se B2W_CSV estiver definido; caso contrário, lê direto do GitHub
URL = os.environ.get("B2W_CSV", "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/main/B2W-Reviews01.csv")
df_full = pd.read_csv(URL, low_memory=False)

# Exibe as dimensões do dataset e uma prévia das 3 primeiras linhas
print(df_full.shape)
df_full.head(3)

Removemos textos vazios e duplicados e trabalhamos com uma **amostra de 20 mil avaliações** para tudo rodar rápido no Colab. A semente fixa (`random_state=42`) garante que todos vejam a mesma amostra.

In [ ]:
# Remove avaliações nulas ou contendo apenas espaços em branco
df = df_full[df_full["review_text"].notna() & (df_full["review_text"].str.strip() != "")]
print("vazios removidos:", len(df_full) - len(df))

# Remove textos duplicados mantendo apenas a primeira ocorrência
df = df.drop_duplicates("review_text")
print("depois de remover duplicados:", len(df))

# Sorteia a amostra de 20 mil avaliações (semente fixa), conta palavras e confere as notas
amostra = df.sample(20_000, random_state=42).reset_index(drop=True)
amostra["n_palavras"] = amostra["review_text"].str.split().str.len()
amostra["overall_rating"].value_counts().sort_index()

In [ ]:
import matplotlib.pyplot as plt

# Cria figura com 2 gráficos lado a lado (notas e comprimento dos textos)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Distribuição de avaliações por nota
amostra["overall_rating"].value_counts().sort_index().plot.bar(ax=ax[0], color="#2F5B8A")
ax[0].set_title("Avaliações por nota"); ax[0].set_xlabel("nota")

# Histograma do número de palavras (corte em 120 para evitar distorção pela cauda longa)
amostra["n_palavras"].clip(upper=120).plot.hist(bins=40, ax=ax[1], color="#B55A17")
ax[1].set_title("Palavras por avaliação (corte em 120)"); ax[1].set_xlabel("palavras")

# Exibe os gráficos e o resumo estatístico do comprimento
plt.tight_layout(); plt.show()
amostra["n_palavras"].describe()

## Bloco 1 · O texto vem sujo

### Problema
Antes de qualquer técnica, **leia** os textos. Rode a célula abaixo algumas vezes e anote o que aparece.

In [ ]:
# Sorteia 8 avaliações aleatórias (sem semente) e imprime nota e início do texto
for r in amostra.sample(8).itertuples():
    print(f"{r.overall_rating}★ | {r.review_text[:200]}")

Alguns padrões podem ser contados com expressões regulares:

In [ ]:
# Detecta padrões comuns de sujeira textual com expressões regulares
sinais = {
    "CAIXA ALTA": amostra["review_text"].str.isupper(),
    "pontuação repetida (!!!, ???)": amostra["review_text"].str.contains(r"[!?]{3,}"),
    "abreviações (vc, q, pq, tb, msm)": amostra["review_text"].str.contains(r"\b(?:vc|q|pq|tb|td|msm)\b", case=False),
    "letras esticadas (ruimmmm)": amostra["review_text"].str.contains(r"([a-z])\1{3,}", case=False),
    "URL": amostra["review_text"].str.contains(r"https?://|www\.", case=False),
}

# Totaliza quantas avaliações apresentam cada tipo de padrão
pd.Series({k: int(v.sum()) for k, v in sinais.items()}, name="avaliações").to_frame()

### A armadilha do "não"
A lista de stopwords do NLTK é o ponto de partida mais comum. Veja o que ela faz com uma reclamação:

In [ ]:
from nltk.corpus import stopwords

# Carrega stopwords padrão do NLTK e verifica se as negações estão incluídas
stop_nltk = set(stopwords.words("portuguese"))
print("'não' está na lista?", "não" in stop_nltk, "| 'nao' está?", "nao" in stop_nltk)

# Demonstra o efeito da lista padrão em uma frase negativa
frase = "O produto não chegou e não recebi resposta"
print([w for w in frase.lower().split() if w not in stop_nltk])

O efeito numa busca é imediato. Com a lista padrão, a consulta *o produto não chegou* vira só `produto chegou`:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
import numpy as np
import pandas as pd

# Garante que a lista de stopwords em português do NLTK esteja carregada
try:
    stop_nltk = set(stopwords.words("portuguese"))
except Exception:
    import nltk
    nltk.download("stopwords", quiet=True)
    stop_nltk = set(stopwords.words("portuguese"))

# Vetoriza a amostra com TF-IDF usando a lista padrão (que descarta "não")
v_padrao = TfidfVectorizer(stop_words=list(stop_nltk))
X_padrao = v_padrao.fit_transform(amostra["review_text"])
print("a consulta vira:", v_padrao.build_analyzer()("o produto não chegou"))

# Calcula similaridade de cosseno e lista as 10 avaliações mais próximas da consulta
sc = cosine_similarity(v_padrao.transform(["o produto não chegou"]), X_padrao).ravel()
pd.DataFrame([{"score": round(sc[i], 2), "nota": amostra.overall_rating[i], "texto": amostra.review_text[i][:100]} for i in np.argsort(-sc)[:10]])

A frase perdeu as duas negações e passou a dizer o contrário. E como muitos clientes escrevem *nao* sem acento, a mesma palavra ainda vira dois tokens. **Pré-processar é decidir.**

### Conceito: as quatro decisões
1. **Tokenização:** onde cortar o texto.
2. **Normalização:** minúsculas, acentos, pontuação repetida, URLs.
3. **Stopwords:** quais palavras saem (e quais precisam ficar).
4. **Lematização:** juntar flexões (*entregue, entregaram → entregar*).

### Solução: um pipeline com decisões explícitas

In [ ]:
import unicodedata
import re

# Stopwords customizadas: adiciona termos genéricos do domínio e preserva negações/contraste
STOP = (stop_nltk | {"produto", "americanas"}) - {"não", "nao", "nem", "sem", "mas"}

def normalizar(texto):
    """Converte para minúsculas, remove URLs, reduz letras repetidas e padroniza abreviações."""
    t = texto.lower()
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)      # URLs
    t = re.sub(r"([a-zà-ú])\1{2,}", r"\1", t)          # ruimmmm -> ruim
    t = re.sub(r"\bnao\b", "não", t)                   # unifica a negação
    t = re.sub(r"\b(vc)\b", "você", t)
    t = re.sub(r"\b(q)\b", "que", t)
    t = re.sub(r"\b(pq)\b", "porque", t)
    return t

def preprocessar(textos, lematizar=True):
    """Normaliza, tokeniza, lematiza (opcional) e filtra stopwords em lote com spaCy."""
    saida = []
    for doc in nlp.pipe((normalizar(t) for t in textos), batch_size=500):
        toks = []
        for tok in doc:
            if tok.is_alpha:
                # o spaCy expande contrações ("dos" -> "de o"): separamos as partes
                toks.extend((tok.lemma_ if lematizar else tok.text).split())
        saida.append(" ".join(w for w in toks if w not in STOP))
    return saida

# Testa o pipeline em uma avaliação real que contém "nao" sem acento
exemplo = amostra.loc[amostra["review_text"].str.contains("nao", case=False)].iloc[0, :]["review_text"]
print("ORIGINAL :", exemplo)
print("NORMAL.  :", normalizar(exemplo))
print("PIPELINE :", preprocessar([exemplo])[0])

Agora aplicamos o pipeline à amostra (leva cerca de um minuto) e medimos o vocabulário a cada etapa.

In [ ]:
%%time
# Aplica o pipeline de pré-processamento à amostra inteira
amostra["texto_limpo"] = preprocessar(amostra["review_text"].tolist())

def tamanho_vocab(textos):
    """Conta quantas palavras distintas (vocabulário) existem em uma coleção de textos."""
    return len({w for t in textos for w in t.split()})

# Mede a redução do vocabulário ao longo das etapas de limpeza
etapas = {
    "bruto (split)": amostra["review_text"].tolist(),
    "minúsculas": amostra["review_text"].str.lower().tolist(),
    "normalizado": [normalizar(t) for t in amostra["review_text"]],
    "pipeline completo": amostra["texto_limpo"].tolist(),
}
pd.Series({k: tamanho_vocab(v) for k, v in etapas.items()}, name="palavras distintas").to_frame()

### Lei de Zipf
Poucas palavras aparecem muito; a maioria aparece pouquíssimo. Em escala log-log a curva vira quase uma reta.

In [ ]:
import collections
import re
import numpy as np
import matplotlib.pyplot as plt

# Conta a frequência das palavras da amostra e ordena do mais ao menos frequente
cont = collections.Counter(w for t in amostra["review_text"].str.lower() for w in re.findall(r"\w+", t))
freqs = np.array(sorted(cont.values(), reverse=True))

# Conta palavras de ocorrência única (hapax legomena) e mostra o top 10
hapax = sum(1 for v in cont.values() if v == 1)
print(f"palavras distintas: {len(cont):,} | aparecem uma única vez: {hapax:,} ({hapax/len(cont):.0%})")
print("10 mais frequentes:", cont.most_common(10))

# Plota ranking x frequência em escala log-log (Lei de Zipf)
plt.loglog(np.arange(1, len(freqs) + 1), freqs, color="#B55A17")
plt.xlabel("posição no ranking (log)"); plt.ylabel("frequência (log)"); plt.title("Lei de Zipf na amostra")
plt.show()

## Bloco 2 · Contar palavras engana

### Problema
Quais palavras aparecem mais nas notas 1 e 5?

In [ ]:
def top_palavras(textos, k=10):
    """Retorna as k palavras mais frequentes nos textos brutos em minúsculas."""
    return collections.Counter(w for t in textos for w in re.findall(r"\w+", t.lower())).most_common(k)

# Compara as 10 palavras brutas mais comuns nas avaliações de nota 1 e nota 5
pd.DataFrame({
    "nota 1": [f"{w} ({n})" for w, n in top_palavras(amostra.loc[amostra.overall_rating == 1, "review_text"])],
    "nota 5": [f"{w} ({n})" for w, n in top_palavras(amostra.loc[amostra.overall_rating == 5, "review_text"])],
})

As listas são quase iguais: a contagem bruta mostra a língua portuguesa, não o problema do cliente.

### Conceito: BoW e TF-IDF
- **Bag-of-Words:** cada documento vira um vetor de contagens, uma coluna por palavra do vocabulário.
- **TF-IDF:** pondera a contagem pela raridade no corpus.

$$\text{TF-IDF}(t,d) = \underbrace{\frac{\text{ocorrências de } t \text{ em } d}{\text{palavras em } d}}_{TF} \times \underbrace{\log_{10}\frac{N}{df(t)}}_{IDF}$$

### Solução: termos que distinguem as notas

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Gera a matriz TF-IDF com unigramas e bigramas presentes em ao menos 5 avaliações
tfidf = TfidfVectorizer(min_df=5, ngram_range=(1, 2))
X = tfidf.fit_transform(amostra["texto_limpo"])
vocab = np.array(tfidf.get_feature_names_out())
print("matriz:", X.shape, f"| densidade: {X.nnz / (X.shape[0] * X.shape[1]):.2%}")

def distintivos(nota, k=15):
    """Encontra os k termos com maior diferença de TF-IDF médio na nota alvo frente às demais."""
    alvo = (amostra["overall_rating"] == nota).values
    dif = np.asarray(X[alvo].mean(0)).ravel() - np.asarray(X[~alvo].mean(0)).ravel()
    return list(vocab[np.argsort(-dif)[:k]])

# Exibe os termos mais característicos das notas 1 e 5
pd.DataFrame({"nota 1": distintivos(1), "nota 5": distintivos(5)})

**Repare:** "entrega" aparece entre as palavras frequentes das duas notas. A mesma palavra serve a "entrega rápida" e a "entrega nunca chegou". Os bigramas (`ngram_range=(1, 2)`) ajudam, mas o limite da representação esparsa continua: ela não sabe que *entregue* e *chegou* falam da mesma coisa.

### Cálculo à mão (treino para a parte individual)

In [ ]:
from collections import Counter

# Calcula o document frequency (df) real de cada termo na amostra pré-processada
N = len(amostra)
df_termo = Counter(w for t in amostra["texto_limpo"] for w in set(t.split()))

# Seleciona uma avaliação curta (8 a 15 palavras) de nota 1 para o cálculo passo a passo
doc = amostra.loc[(amostra["overall_rating"] == 1) & amostra["texto_limpo"].str.split().str.len().between(8, 15), "texto_limpo"].iloc[0]
palavras = doc.split()
print("documento:", doc, "\n")

# Calcula TF, IDF (log10) e TF-IDF para cada termo único do documento
linhas = []
for termo in sorted(set(palavras)):
    tf = palavras.count(termo) / len(palavras)
    idf = np.log10(N / df_termo[termo])
    linhas.append((termo, palavras.count(termo), df_termo[termo], round(tf, 3), round(idf, 3), round(tf * idf, 4)))
pd.DataFrame(linhas, columns=["termo", "vezes no doc", "df", "TF", "IDF", "TF-IDF"]).sort_values("TF-IDF", ascending=False)

## Bloco 3 · Mesmo problema, palavras diferentes

### Problema
Duas reclamações reais da amostra:

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Duas reclamações com o mesmo sentido, mas vocabulário diferente
a = "O pedido ainda não foi entregue.  Muito demorado. Estou no aguardo ainda."
b = "Continuo aguardando a justificativa para tanta demora de entrega."

# Similaridade de cosseno usando TF-IDF no texto bruto
tf_ab = TfidfVectorizer().fit(amostra["review_text"])
print("similaridade TF-IDF (texto bruto):", round(cosine_similarity(tf_ab.transform([a]), tf_ab.transform([b]))[0, 0], 2))

# Repete a comparação após pré-processamento e verifica termos em comum
la, lb = preprocessar([a, b])
print("lematizadas:", la, "|", lb)
print("similaridade TF-IDF (lematizado):", round(cosine_similarity(tfidf.transform([la]), tfidf.transform([lb]))[0, 0], 2))
print("palavras em comum depois de lematizar:", set(la.split()) & set(lb.split()) or "nenhuma")

### Conceito: embeddings
Um modelo de embeddings transforma cada texto num **vetor denso** (aqui, 384 números) treinado para que textos com significado parecido fiquem próximos. A proximidade é medida pelo **cosseno** entre os vetores.

Usamos o modelo multilíngue `paraphrase-multilingual-MiniLM-L12-v2`, pequeno o suficiente para rodar em CPU.

In [ ]:
from sentence_transformers import SentenceTransformer

# Carrega o modelo multilíngue e gera vetores densos (384D) normalizados para as duas frases
modelo = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
ea, eb = modelo.encode([a, b], normalize_embeddings=True)

# Com vetores de norma 1, o produto escalar (@) equivale à similaridade de cosseno
print("similaridade de cosseno com embeddings:", round(float(ea @ eb), 2))

Agora a amostra inteira. Em CPU leva alguns minutos; com GPU no Colab (Ambiente de execução → Alterar tipo → T4), segundos.

In [ ]:
%%time
from sentence_transformers import SentenceTransformer

# Garante que o modelo de embeddings esteja carregado em memória
try:
    modelo
except NameError:
    modelo = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Gera a matriz densa de embeddings normalizados da amostra em lotes de 128 textos
E = modelo.encode(amostra["review_text"].tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=True)
E.shape

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

### Solução: busca semântica

In [ ]:
def buscar(consulta, k=5):
    """Retorna lado a lado os k resultados mais próximos da consulta em TF-IDF e embeddings."""
    # Representa a consulta no mesmo espaço vetorial dos documentos
    q_t = tfidf.transform(preprocessar([consulta]))
    q_e = modelo.encode([consulta], normalize_embeddings=True)

    # Calcula similaridade de cosseno em cada método e reúne os top k resultados
    res = []
    for nome, sc in [("TF-IDF", cosine_similarity(q_t, X).ravel()), ("embeddings", (E @ q_e.T).ravel())]:
        for i in np.argsort(-sc)[:k]:
            res.append({"método": nome, "score": round(float(sc[i]), 2), "nota": amostra.overall_rating[i], "texto": amostra.review_text[i][:120]})
    return pd.DataFrame(res)

# Testa a busca comparativa com uma consulta negativa sobre entrega
buscar("o produto não chegou")

In [ ]:
# Testa a busca comparativa com uma consulta sobre produto avariado
buscar("chegou quebrado")

**Leitura crítica:** os embeddings encontram sinônimos ("não foi entregue", "não recebi"), mas também erram, trazendo às vezes uma reclamação de outro tipo. E textos em CAIXA ALTA tendem a ficar próximos entre si, qualquer que seja o assunto: a normalização importa também para embeddings.

### O mapa do espaço
Projetamos 5 mil vetores em 2D com UMAP. Cada ponto é uma avaliação; a cor é a nota.

In [ ]:
import umap

# Sorteia 5 mil avaliações (semente fixa) e reduz os embeddings (384D -> 2D) com UMAP
idx = np.random.RandomState(0).choice(len(amostra), 5000, replace=False)
coords = umap.UMAP(n_neighbors=30, min_dist=0.1, metric="cosine", random_state=42).fit_transform(E[idx])

# Plota a projeção 2D colorindo cada avaliação de acordo com sua nota
cores = {1: "#9C4A12", 2: "#E8924F", 3: "#B9B4A8", 4: "#7FA7CF", 5: "#2F5B8A"}
plt.figure(figsize=(8, 7))
plt.scatter(coords[:, 0], coords[:, 1], s=4, c=[cores[n] for n in amostra.overall_rating.values[idx]], alpha=0.7)
plt.title("5 mil avaliações no espaço de embeddings (cor = nota)"); plt.axis("off"); plt.show()

### Outras medidas de similaridade
O cosseno serve para vetores densos. Para outros problemas, outras medidas:
- **Jaccard:** sobreposição de conjuntos de palavras. Boa para achar textos quase duplicados.
- **Levenshtein:** número de edições entre duas strings. Boa para erros de digitação e nomes de produtos.

In [ ]:
def jaccard(x, y):
    """Calcula a similaridade de Jaccard entre os conjuntos de palavras de dois textos."""
    A, B = set(x.lower().split()), set(y.lower().split())
    return len(A & B) / len(A | B)

def levenshtein(s, t):
    """Calcula a distância de edição (Levenshtein) entre duas strings via programação dinâmica."""
    ant = list(range(len(t) + 1))
    for i, cs in enumerate(s, 1):
        atual = [i]
        for j, ct in enumerate(t, 1):
            atual.append(min(ant[j] + 1, atual[j - 1] + 1, ant[j - 1] + (cs != ct)))
        ant = atual
    return ant[-1]

# Demonstra o cálculo das duas medidas em pares de exemplo
print("Jaccard:", round(jaccard("produto chegou no prazo recomendo", "chegou no prazo recomendo o produto"), 2))
print("Levenshtein smartfone x smartphone:", levenshtein("smartfone", "smartphone"))

---
# Etapa 1 do projeto · entrega até 23/11, 23h59

A partir daqui, o notebook é de vocês. As células abaixo seguem os **7 itens do checklist** apresentado no laboratório, na mesma ordem e com a mesma numeração.

**Como entregar**
1. Salvem uma cópia deste notebook no repositório do grupo (*Arquivo › Salvar uma cópia no GitHub*).
2. Preencham cada item: as células de código têm comentários com o que fazer; as células **Resposta do grupo** são para o texto.
3. Rodem o notebook inteiro antes de entregar, para que as saídas fiquem salvas.
4. Entreguem o link do repositório, com um README curto (integrantes e onde usaram IA, se usaram).

| # | Item | Critério de aceite |
|---|---|---|
| 1 | Amostra | Amostra própria (outra semente e outro tamanho), justificada em um parágrafo |
| 2 | Exploração | Notas, comprimento, vazios e duplicados, com gráficos |
| 3 | Catálogo de sujeira | 30 avaliações lidas e a tabela preenchida |
| 4 | Pipeline | Cada decisão justificada e um exemplo do que a limpeza apagaria |
| 5 | Representações | TF-IDF e embeddings da amostra, com a densidade da matriz |
| 6 | Busca | Três consultas do grupo, top 5 dos dois métodos e análise |
| 7 | Individual | TF-IDF à mão de três termos e uma decisão explicada |

Itens 1 a 6: grupo (10 pontos). Item 7: individual (5 pontos).

## Item 1 · Amostra

**O que fazer:** criar a amostra **do grupo**, que será usada no projeto inteiro.

**Obrigatório:** a amostra não pode ser a da aula.
- **Semente:** escolham uma diferente de 42 e guardem na variável `SEMENTE`. Usem a mesma em todo o notebook.
- **Tamanho:** diferente de 20 mil. Entre 5 mil e 30 mil costuma rodar bem no Colab.
- **Critério:** livre, desde que justificado. Pode ser aleatória, estratificada por nota (mesma quantidade de cada nota) ou por categoria de produto.

Todos os itens seguintes (2 a 7) usam esta amostra: a variável `amostra` é substituída aqui.

**Critério de aceite:** amostra própria (outra semente e outro tamanho), justificada em um parágrafo.

In [ ]:
# Define a semente fixa do grupo (diferente de 42)
SEMENTE = 50

# Cria amostra estratificada com 5 mil avaliações por nota (total: 25 mil) e conta palavras
amostra = df.groupby("overall_rating").sample(5_000, random_state=SEMENTE).reset_index(drop=True)
amostra["n_palavras"] = amostra["review_text"].str.split().str.len()

# Valida os requisitos de amostra própria e exibe dimensões e distribuição das notas
assert SEMENTE != 42 and len(amostra) != 20_000, "A amostra precisa ser diferente da usada em aula"
print(amostra.shape); amostra["overall_rating"].value_counts().sort_index()

**Resposta do grupo (Item 1):** qual a semente, o tamanho e o critério? Por que essa escolha atende ao problema (por exemplo, equilíbrio entre notas, tempo de execução no Colab, foco em reclamações)?

_A semente foi 50, o tamanho ficou 25000 amostras, 5 mil para cara nota. Atende o problema pois teremos um equilibrio de respostas positivas e negativas, evitando um vies na amostra, é um número de amostras bom para o colab processar sem problema._

## Item 2 · Exploração

**O que fazer:** descrever a amostra com números e gráficos.

- Distribuição das notas e da recomendação (`recommend_to_a_friend`).
- Comprimento dos textos (em palavras).
- Quantos textos vazios e duplicados existiam no dataset original.
- A seção 1 já tem parte disso: podem reaproveitar o código, mas comentem o que observam.

**Critério de aceite:** notas, comprimento, vazios e duplicados, com gráficos.

In [ ]:
# Cria figura vertical com 3 gráficos exploratórios da amostra do grupo
fig, ax = plt.subplots(3, 1, figsize=(12, 12))

# Distribuição das notas (1 a 5)
amostra["overall_rating"].value_counts().sort_index().plot.bar(ax=ax[0], color="#2F5B8A")
ax[0].set_title("Avaliações por nota"); ax[0].set_xlabel("nota")

# Distribuição da recomendação a um amigo (Yes/No)
amostra["recommend_to_a_friend"].value_counts().plot.bar(ax=ax[1], color=["#2F5B8A", "#B55A17"], rot=0)
ax[1].set_title("Recomendaria a um amigo? (amostra)"); ax[1].set_xlabel("")

# Histograma do comprimento das avaliações (corte em 120 palavras)
amostra["n_palavras"].clip(upper=120).plot.hist(bins=40, ax=ax[2], color="#B55A17")
ax[2].set_title("Palavras por avaliação (corte em 120)"); ax[2].set_xlabel("palavras")

# Média e mediana de palavras por nota
amostra.groupby("overall_rating")["n_palavras"].agg(["mean", "median"]).round(1)

# Resumo de registros nulos e duplicados no dataset original antes da limpeza
texto = df_full["review_text"]
validos = texto[texto.notna() & (texto.str.strip() != "")]

resumo = pd.Series({
    "linhas no dataset original": len(df_full),
    "Linhas da amostra original": len(amostra),
    "texto nulo": texto.isna().sum(),
    "cópias de textos repetidos": validos.duplicated().sum(),
    "linhas após a limpeza (df)": len(df),
})
display(resumo.to_frame("quantidade"))

**Resposta do grupo (Item 2):** o que os números mostram? As notas estão equilibradas? As avaliações são curtas ou longas? Isso muda alguma decisão daqui para a frente?

_Mostram que não temos muitas linhas nulos ou duplicados. Sim, pois fizemos a divisão da amostra por nota de maneira equilibrada, 5 mil avaliações por nota. Em sua maioria as availações utilizam poucas palavras, curta. Sim, sabemos que temos que fazer a limpeza desse dataset,removendo as avaliações nulas e duplicas, sabemos também que não iremos ter um vies das notas, pois estão equibilibradas, notamos também que apesar das notas estarem equilibradas, no geral, as pessoas recomendam mais do que não, provavelmente as pessoas que dão nota 3 a 5 recomendam, enquanto pessoas que dão nota 1 ou 2 não recomendam._

## Item 3 · Catálogo de sujeira

**O que fazer:** ler 30 avaliações e registrar os problemas encontrados.

1. Sorteiem 30 avaliações da amostra do grupo, com a `SEMENTE` do Item 1.
2. Leiam todas, com atenção.
3. Para cada tipo de problema, registrem: o problema, um exemplo real, em quantas das 30 ele aparece e a **decisão** do grupo.

O que procurar: abreviações, caixa alta, pontuação repetida, letras esticadas, erros de digitação, emojis, números e preços, e **dados pessoais** (nomes, telefones, e-mails).

**Critério de aceite:** 30 avaliações lidas e a tabela preenchida. A coluna **decisão** é a que mais vale: ela precisa dizer o que se ganha e o que se perde, não apenas "remover".

In [ ]:
# Sorteia e imprime 30 avaliações da amostra do grupo para leitura manual
lidas = amostra.sample(30, random_state=50)
for r in lidas.itertuples():
  print(f"{r.overall_rating}★ | {r.review_text}")

# Consolida os tipos de sujeira identificados nas 30 avaliações e a decisão para cada um
catalogo = pd.DataFrame([
    {
        "problema": "Erros de digitação",
        "exemplo real": "mrecebi",
        "quantas em 30": 4,
        "decisão": "Corrigir erros evidentes. Ganha padronização, mas pode alterar palavras se a correção for incorreta."
    },
    {
        "problema": "Abreviações",
        "exemplo real": "vc / hj",
        "quantas em 30": 2,
        "decisão": "Substituir abreviações conhecidas. Ganha uniformidade, mas perde características da linguagem informal."
    },
    {
        "problema": "Caixa alta",
        "exemplo real": "bOA TARDE",
        "quantas em 30": 3,
        "decisão": "Converter para minúsculas. Ganha padronização, mas perde ênfase."
    },
    {
        "problema": "Ausência de acentuação",
        "exemplo real": "rapidos / Nao",
        "quantas em 30": 3,
        "decisão": "Corrigir acentuação. Ganha consistência, mas exige cuidado com o significado."
    },
    {
        "problema": "Espaços duplicados",
        "exemplo real": "chegou  antes",
        "quantas em 30": 3,
        "decisão": "Remover espaços extras. Ganha organização sem perda significativa."
    }
])

catalogo

**Resposta do grupo (Item 3):** qual problema foi o mais frequente? Algum surpreendeu? Encontraram dados pessoais?


Uns dos problemas mais frequente foram os erros de ortografia.Sim,letras em maiúsculo e minúsculo misturadas.Não.

## Item 4 · Pipeline

**O que fazer:** adaptar o pré-processamento ao que o catálogo mostrou.

- Copiem para a célula abaixo a função `normalizar` e o conjunto `STOP` (Bloco 1) e alterem o que for preciso. Cada alteração deve vir de uma linha do catálogo.
- Reapliquem o pipeline à amostra e meçam o vocabulário antes e depois.
- Mostrem **um exemplo de algo que a limpeza apaga e não deveria apagar** (uma avaliação antes e depois, com o que se perdeu).

**Critério de aceite:** cada decisão justificada e um exemplo do que a limpeza apagaria.

In [ ]:
import unicodedata

# Stopwords do grupo: remove termos genéricos e preserva palavras de negação/contraste
STOP = (stop_nltk | {"produto", "americanas"}) - {"não", "nao", "nem", "sem", "mas"}

def normalizar(texto):
    """Aplica as regras de limpeza do grupo: minúsculas, URLs, repetições, abreviações, acentos e espaços."""
    t = texto.lower()
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)      # URLs
    t = re.sub(r"([a-zà-ú])\1{2,}", r"\1", t)          # ruimmmm -> ruim
    t = re.sub(r"\bnao\b", "não", t)                   # unifica a negação
    t = re.sub(r"\bhj\b", "hoje", t)                   # abreviação do catálogo
    t = re.sub(r"\bmrecebi\b", "recebi", t)            # erro de digitação do catálogo
    t = re.sub(r"\b(vc)\b", "você", t)
    t = re.sub(r"\b(q)\b", "que", t)
    t = re.sub(r"\b(pq)\b", "porque", t)
    t = re.sub(r"\brapidos\b", "rápidos", t)           # acentuação do catálogo
    t = re.sub(r"\bate\b", "até", t)                   # acentuação do catálogo
    t = re.sub(r"\s+", " ", t).strip()                  # remove espaços duplicados
    return t

def preprocessar(textos, lematizar=True):
    """Normaliza, tokeniza, lematiza (opcional) e filtra stopwords em lote usando spaCy."""
    saida = []
    for doc in nlp.pipe((normalizar(t) for t in textos), batch_size=500):
        toks = []
        for tok in doc:
            if tok.is_alpha:
                # o spaCy expande contrações ("dos" -> "de o"): separamos as partes
                toks.extend((tok.lemma_ if lematizar else tok.text).split())
        saida.append(" ".join(w for w in toks if w not in STOP))
    return saida

# Valida o efeito das etapas do pipeline em uma avaliação de exemplo
exemplo = amostra.loc[amostra["review_text"].str.contains("nao", case=False)].iloc[0, :]["review_text"]
print("ORIGINAL :", exemplo)
print("NORMAL.  :", normalizar(exemplo))
print("PIPELINE :", preprocessar([exemplo])[0])

# Reaplica o pipeline do grupo à amostra inteira
amostra["texto_limpo"] = preprocessar(amostra["review_text"].tolist())

# Mede o tamanho do vocabulário antes e depois da limpeza
def tamanho_vocab(textos):
    """Conta quantas palavras distintas existem na coleção de textos."""
    return len({w for t in textos for w in t.split()})

etapas = {
    "bruto (split)": amostra["review_text"].tolist(),
    "minúsculas": amostra["review_text"].str.lower().tolist(),
    "normalizado": [normalizar(t) for t in amostra["review_text"]],
    "pipeline completo": amostra["texto_limpo"].tolist(),
}
pd.Series({k: tamanho_vocab(v) for k, v in etapas.items()}, name="palavras distintas").to_frame()

# Exibe um caso real onde a limpeza apaga informações relevantes (datas e intensificadores)
i = 5
print("ORIGINAL:", amostra.review_text[i]); print("LIMPO   :", amostra.texto_limpo[i])

**Resposta do grupo (Item 4):** liste cada decisão do pipeline com a justificativa (o que ganha, o que perde). Depois, explique o exemplo: o que a limpeza apagou e por que isso importava?

| Decisão | Justificativa |
|---|---|
| _exemplo: manter "não", "nem" e "sem" fora das stopwords_ | _a negação inverte o sentido da avaliação_ |
|  |  |
conversão para letras minúsculas,remoção de URLs,correção de erros de digitação,Substituição de abreviações,correção da acentuação,redução de letras repetidas,remoção de espaços duplicados,remoção de stopwords automática ,preservação de negações e lematização.


-Conversão para letra em minúsculas:solução, transforma todas as letras em minúsculas. as palavras ficam padronizadas.Mas perde a ênfase(sentimento,emoção) que o consumidor pode expressar usando letras maiúsculas.Ex:bOA TARDE-> boa tarde.


-Remoção de URLs:solução, elimina links presentes nas avaliações, ganha organização e reduz informações que geralmente não são relevantes para a análise. Mas perde possíveis referências a páginas ou produtos compartilhados pelo consumidor.

-Correção de erros de digitação:solução,corrige os erros identificados nas avaliações,a escrita fica mais uniforme.Mas pode perder ou alterar o sentido caso a correção seja feita incorretamente.Ex:mrecebi → recebi.

-Substituição de abreviações:solução, padroniza palavras escritas informalmente,ganha consistência no vocabulário, mas perde características da linguagem informal utilizada pelos consumidores,mas não é uma grande perde para nosso contexto.Ex:hj → hoje.

-Correção da acentuação :solução, padroniza a escrita,ganha consistência e facilita o processamento das palavras.Mas exige cuidado para não alterar o significado.Ex:nao → não.

-Redução de letras repetidas : solução,reduz repetições exageradas,ganha padronização, mas perde parte da emoção e da intensidade expressa pelo consumidor.Ex:boooom → bom

-Remoção de espaços duplicados:solução, substitui vários espaços por apenas um,ganha organização sem perda significativa de informação.Ex:chegou  antes-> chegou antes

-Remoção de stopwords:soulução, elimina palavras consideradas pouco relevantes para a análise, como artigos e palavras frequentes, ganha redução do vocabulário e simplificação dos textos, mas pode perder informações importantes quando uma palavra removida tem significado para a avaliação.Ex:o, os,as..

-Preservação de negações:solução, essas palavras foram mantidas fora da lista de remoção porque podem alterar o sentido da opinião do consumidor,ganha preservação do significado, evitando que uma reclamação seja interpretada como elogio ou vice-versa.Ex:não, nem, sem e mas

-Lematização:soulçaõ, transforma palavras em suas formas básicas, ganha redução de variações das palavras e padronização do vocabulário, mas perde algumas diferenças de tempo verbal e forma de expressão.Ex:chegou -> chegar

Ex de uma avaliação:

ORIGINAL: Efetuei a compra no dia 24/04 e até agora no dia 02/05 não foi emitido nem a nota fiscal??? Tenho certeza que será um ótimo produto, mas a loja deixou muito a desejar. E quanto comprei o prazo de entrega para onde moro era um, hoje quando fui ver diminuíram, apenas na simulação de frente, mas no meu pedido não.
LIMPO   : efetuei compra dia agora dia não emitir nem nota fiscal ter certeza bom mas loja deixar desejar quanto comprei prazo entrega onde moro hoje ver diminuir apenas simulação frente mas pedido não
Oque foi perdido na frase ?, a data que a compra foi realizada e a data que a nota fiscal foi emitida, perdendo a informação sobre o tempo ,com isso,não conseguindo identificar o período da demora da emissão ,também foi removida a palavra "muito" perdendo a instensidade da insastifição do cliente.Dito isso perdemos informações importantes para compreeneder a reclamação.

## Item 5 · Representações

**O que fazer:** representar a amostra (já com o pipeline do grupo) de duas formas.

- **TF-IDF** sobre `texto_limpo`, com os parâmetros que o grupo escolher (`min_df`, `ngram_range`).
- **Embeddings** sobre o texto original (`review_text`), com o modelo do Bloco 3.
- Reportem o formato das duas matrizes e a **densidade** da matriz TF-IDF.
- Salvem os embeddings (`np.save`), de preferência no Google Drive: eles voltam na Etapa 2.

**Critério de aceite:** TF-IDF e embeddings da amostra, com a densidade da matriz.

In [ ]:
# Gera a representação TF-IDF da amostra do grupo e calcula a densidade da matriz esparsa
tfidf = TfidfVectorizer(min_df=5, ngram_range=(1, 2))
X = tfidf.fit_transform(amostra["texto_limpo"])
print("TF-IDF:", X.shape, f"| densidade: {X.nnz / (X.shape[0] * X.shape[1]):.2%}")

# Gera os embeddings densos (384D) sobre o texto original e salva em disco para a Etapa 2
E = modelo.encode(amostra["review_text"].tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=True)
print("embeddings:", E.shape)
np.save("embeddings.npy", E)

**Resposta do grupo (Item 5):** quais parâmetros usaram no TF-IDF e por quê? Comparem as duas representações: tamanho, densidade e o que cada uma consegue ou não capturar.

Nós usamos min df=5 e  ngram range=1,2,nós desconsideramos palavra que aparerecem em menos de 5 avaliações,com isso redurzimos termos muito raros e possíveis rúidos.Já no range escolhemos" 1,2" por ser o meio termo ,porque permite considerar tanto palavras isoladas quanto combinações de duas palavras.
O TF-IDF possui uma coluna para cada palavra podendo ter milhares de colunas,ele dá mais importância às palavras que aparecem nas avaliações e ajuda a identificar os termos mais relevantes.Trabalham principalmente com palavras e suas frequências, mas tem dificuldade para entender o sentido de uma frase e o contexto em que ela foi escrita. Além disso, sua matriz pode possuir muitos valores iguais a zero, pois nem todas as palavras aparecem em todas as avaliações.Os Embeddings possui uma quantidade fixa de dimensões(384), vão além das palavras isoladas, pois procuram representar o significado dos textos. Assim, conseguem identificar avaliações com sentidos parecidos, mesmo quando as palavras utilizadas são diferentes. Os seus resultados são mais difíceis de interpretar diretamente.
As duas técnicas se complementam o TF-IDF ajuda a identificar os termos mais importantes das avaliações, enquanto os Embeddings ajudam a encontrar semelhanças de significado entre elas.

## Item 6 · Busca

**O que fazer:** comparar TF-IDF e embeddings em três consultas escolhidas pelo grupo.

- Sugestão: uma consulta sobre **entrega**, uma sobre **defeito** e uma sobre **atendimento**.
- Usem a função `buscar()` do Bloco 3: ela mostra os 5 primeiros resultados de cada método lado a lado.
- **Leiam** os resultados: score alto não significa acerto.

**Critério de aceite:** três consultas do grupo, top 5 dos dois métodos e análise.

In [ ]:
# Executa as 3 consultas escolhidas pelo grupo comparando o top 5 de TF-IDF e embeddings
for consulta in ["produto entregue", "produto veio com defeito", "atendimento da loja"]:
      display(buscar(consulta))

**Resposta do grupo (Item 6):** para cada consulta, quantos dos 5 resultados de cada método são realmente relevantes? Onde cada método acertou e onde errou? Mostrem exemplos. Qual método o grupo usaria no projeto, e por quê?

| Consulta | Relevantes TF-IDF (de 5) | Relevantes embeddings (de 5) |
|---|---|---|
|  produto entregue  |  3 |  4|
| produto veio com defeito |  5| 4 |
| atendimento da loja | 4 | 3 |

no "produto entregue":

em TF-IDF,como queremos saber avaliações só o produto **entregue** ,foi encontrado 3 avaliações que confirmavam a entrega do produto, mas os outros 2 foi estavam aguardando a entrega ou que não haviam recebido o produto, está dentro do contexto mas não foi relevante nesse caso.
Já no Emebeddings,foi encontrado 4 avaliações relacionadas à entrega, incluindo situações de prazo e recebimento,mas 1 foi mencionando a qualidade da entrega ou o envio, sem confirmar diretamente que o produto havia sido recebido, está parcialmente certo, mas na dúvida deixamos ela de fora.


no "produto veio com defeito"

em TF-IDF consideramos que todas foram relevantes porque apresentaram reclamações relacionadas a defeitos.
Já no Embeddings a 4 avaliações foi decrevendo tipos de problemas encontrados no produto, pórem 1 dos resultados mencionou que o produto tinha uma qualidade ruim ,não que estava com defeito.

no "atendimento da loja"

em TF-IDF foi encontrado 4 avaliações que elogiavam diretamente o atendimento da loja e dos vendedores, pórem 1 dos resultados falava do recebimento de avisos durante a compra, sem avaliar diretamente o atendimento.
Já no Embeddings 3 das avaliações mencionavam o atendimento do fornecedor e da loja, mas as outras 2 apresentou resultados relacionados à entrega, à embalagem e ao site, sem abordar diretamente o atendimento.

De acordo com esses resultados, o que apresentou melhor desempenho foi o TF-IDF e foi o escolhido.




## Item 7 · Individual (5 pontos)

Cada integrante faz a sua parte na subseção com o seu nome. Grupos menores apagam as subseções que sobrarem.

**A · TF-IDF à mão:** escolham uma avaliação **já pré-processada** (diferente da usada em aula) e três termos dela. Para cada termo, mostrem a conta, não só o resultado:

- TF = vezes no documento ÷ palavras no documento
- IDF = log10(N ÷ df), com N = tamanho da amostra e o **df real** da amostra (dicionário `df_termo`, do Bloco 2)
- TF-IDF = TF × IDF

**B · Uma decisão explicada:** um parágrafo sobre uma decisão de pré-processamento do grupo: o que ela ganha, o que ela perde e por que vale a pena no nosso problema.

**Critério de aceite:** TF-IDF à mão de três termos e uma decisão explicada.

In [ ]:
# Recalcula o document frequency (df) real na amostra do grupo após o pipeline do Item 4
df_termo = Counter(w for t in amostra["texto_limpo"] for w in set(t.split()))
N = len(amostra)

# Extrai os dados de apoio (ocorrências no doc e df) para a avaliação escolhida (Integrante 1)
i = 10
palavras = amostra.texto_limpo[i].split()
print(amostra.texto_limpo[i], "| palavras:", len(palavras))
for termo in ["espeto", "jardim", "queimar"]:
        print(termo, "| vezes:", palavras.count(termo), "| df:", df_termo[termo])

### Integrante 1 · _Gabrielle Santana_

**A · TF-IDF à mão**

Avaliação (pré-processada): _espeto jardim menos semana queimar ._ · N = _25.000 _ · palavras no documento = _5_

| Termo | Vezes | df | TF (conta) | IDF (conta) | TF-IDF |
|---|---|---|---|---|---|
| espeto |1  | 5 |1/5=0.2    |log10(25.000/5)=3.6990  |0.7398
| jardim | 1 | 2 |1/5=0.2    |log10(25.000/2)=4.0969  |0.8194
| queimar |1 |103| 1/5=0.2  |log10(25.000/103)=2.3904  |0.4770  |
i=10

**B · Uma decisão explicada**

_Uma das decisões foi transformas todas as palavras das avaliações em letras minúsculas,ajudando na padronização dos textos,para palavras escritas de maneiras diferentes como"bOA TARDE", "boa tarde", "Boa tarde" sejam reconhecidas como a mesma palavra no sistema,evitando a duplicidade do texto  ._

### Integrante 2 · _Marcos_

**A · TF-IDF à mão**

Avaliação (pré-processada): _..._ · N = _..._ · palavras no documento = _..._

| Termo | Vezes | df | TF (conta) | IDF (conta) | TF-IDF |
|---|---|---|---|---|---|
|  |  |  |  |  |  |
|  |  |  |  |  |  |
|  |  |  |  |  |  |

**B · Uma decisão explicada**

_Escrevam aqui._

### Integrante 3 · Tiago

In [ ]:
# Extrai os dados de apoio (ocorrências no doc e df) para a avaliação escolhida
i_tiago = 16
palavras_tiago = amostra.texto_limpo[i_tiago].split()
print("ORIGINAL :", amostra.review_text[i_tiago])
print("LIMPO    :", amostra.texto_limpo[i_tiago], "| palavras:", len(palavras_tiago))
for termo in ["urgente", "chegar", "não"]:
    print(f"{termo} | vezes: {palavras_tiago.count(termo)} | df: {df_termo[termo]}")

**A · TF-IDF à mão**

Avaliação original (`i = 16`): "O produto ainda não chegou, não data estimada para chegar, preciso de uma solução URGENTE."

Avaliação (pré-processada): ainda não chegar não datar estimar chegar preciso solução urgente · N = 25.000 · palavras no documento = 10

| Termo | Vezes | df | TF (conta) | IDF (conta) | TF-IDF |
|---|---|---|---|---|---|
| urgente | 1 | 29 | 1 / 10 = 0,1 | log10(25.000 / 29) = log10(862,0690) = 2,9355 | 0,1 × 2,9355 = **0,2936** |
| chegar | 2 | 3.114 | 2 / 10 = 0,2 | log10(25.000 / 3.114) = log10(8,0283) = 0,9046 | 0,2 × 0,9046 = **0,1809** |
| não | 2 | 10.336 | 2 / 10 = 0,2 | log10(25.000 / 10.336) = log10(2,4187) = 0,3836 | 0,2 × 0,3836 = **0,0767** |

**B · Uma decisão explicada**

Uma decisão central do nosso pipeline foi preservar as palavras de negação e contraste (`"não"`, `"nem"`, `"sem"`, etc) fora da lista de stopwords e unificar a grafia sem acento `"nao"` em `"não"`**. Com isso, ganhamos a preservação do sentido real da avaliação, evitando que reclamações negativas se misturem com positivas. Por outro lado, isso pode gerar ruído em buscas de palavras isoladas. Ainda assim, vale a pena pois separar positivo de negativo é essencial para a busca de classificação e priorização de reclamações.